In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import hashlib


target_pekerjaan = ["Data Analyst", "Backend Engineer", "Full Stack Engineer"]

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
    'Referer': 'https://www.google.com/'
}

In [6]:
test_keyword = "Data Analyst"
test_url = f"https://id.linkedin.com/jobs/search?keywords={test_keyword.replace(' ', '+')}&location=Indonesia"

print(f"Mencoba mengakses: {test_url}")
response = requests.get(test_url, headers=headers)

print(f"Status Code: {response.status_code}")

if response.status_code == 200:
    print("✅ Berhasil! Web bisa diakses.")
    soup = BeautifulSoup(response.text, 'html.parser')
    print("Snippet HTML yang didapat:\n", soup.prettify()[:500])
elif response.status_code == 999:
    print("❌ DIBLOKIR! LinkedIn mendeteksi bot (Error 999). Butuh Selenium untuk menembus ini.")
else:
    print(f"⚠️ Gagal mengakses web. Error code: {response.status_code}")

Mencoba mengakses: https://id.linkedin.com/jobs/search?keywords=Data+Analyst&location=Indonesia
Status Code: 200
✅ Berhasil! Web bisa diakses.
Snippet HTML yang didapat:
 <!DOCTYPE html>
<html lang="in">
 <head>
  <meta content="d_jobs_guest_search" name="pageKey"/>
  <meta content="max-image-preview:large, noarchive" name="robots"/>
  <meta content="max-image-preview:large, archive" name="bingbot"/>
  <!-- -->
  <meta content="urlType=jserp_custom;emptyResult=false" name="linkedin:pageTag"/>
  <meta content="in_ID" name="locale"/>
  <!-- -->
  <meta data-app-version="2.0.2808" data-browser-id="6cd55f1a-5e64-4202-8cfe-fffa18bbd86e" data-call-tree-id="AAZMUma9NZ/q


In [9]:
data_lowongan = []
id_terkumpul = set()

for pekerjaan in target_pekerjaan:
    print(f"\n🔍 Mencari data untuk: {pekerjaan}...")
    url_search = f"https://id.linkedin.com/jobs/search?keywords={pekerjaan.replace(' ', '+')}&location=Indonesia"

    respons_search = requests.get(url_search, headers=headers)
    soup_search = BeautifulSoup(respons_search.text, 'html.parser')

    kartu_pekerjaan = soup_search.find_all('div', class_='base-card')

    for kartu in kartu_pekerjaan:
        try:
            judul = kartu.find('h3', class_='base-search-card__title').text.strip()
            perusahaan = kartu.find('h4', class_='base-search-card__subtitle').text.strip()
            lokasi = kartu.find('span', class_='job-search-card__location').text.strip()

            link_pekerjaan = kartu.find('a', class_='base-card__full-link')['href']

            string_id = f"{judul}_{perusahaan}".encode('utf-8')
            job_id = hashlib.md5(string_id).hexdigest()

            if job_id in id_terkumpul:
                continue
            id_terkumpul.add(job_id)

            print(f"  -> Mengambil detail: {judul} di {perusahaan}...")

            time.sleep(2)
            res_detail = requests.get(link_pekerjaan, headers=headers)
            soup_detail = BeautifulSoup(res_detail.text, 'html.parser')

            kriteria_items = soup_detail.find_all('li', class_='description__job-criteria-item')

            job_type = "Tidak tertera"
            experience_level = "Tidak tertera"

            for item in kriteria_items:
                header_kriteria = item.find('h3', class_='description__job-criteria-subheader').text.strip()
                isi_kriteria = item.find('span', class_='description__job-criteria-text').text.strip()

                if "Employment type" in header_kriteria:
                    job_type = isi_kriteria
                elif "Seniority level" in header_kriteria:
                    experience_level = isi_kriteria

            deskripsi_div = soup_detail.find('div', class_='show-more-less-html__markup')
            if deskripsi_div:
                deskripsi_lengkap = deskripsi_div.text.strip().replace('\n', ' ')
            else:
                deskripsi_lengkap = "Gagal mengambil deskripsi (Mungkin diblokir login wall)"

            job_data = {
                'id': job_id,
                'job_title': judul,
                'company_name': perusahaan,
                'location': lokasi,
                'job_type': job_type,
                'experience_level': experience_level,
                'education_req': "Cek di kolom requirements", # Sering digabung di deskripsi
                'salary_range': "Tidak tertera (LinkedIn jarang publikasi gaji)",
                'job_requirements': deskripsi_lengkap[:500] + "...", # Mengambil 500 karakter pertama
                'job_responsibilities': deskripsi_lengkap[500:1000] + "...", # Mengambil teks selanjutnya
                'posted_date': "Tidak tertera",
                'source_platform': "LinkedIn"
            }
            data_lowongan.append(job_data)

        except Exception as e:
            pass

    time.sleep(3)

print(f"\n✅ Scraping detail selesai! Mengumpulkan {len(data_lowongan)} data.")


🔍 Mencari data untuk: Data Analyst...
  -> Mengambil detail: Data Analyst, Business Intelligence di Shopee...
  -> Mengambil detail: Data Analyst di Traveloka...
  -> Mengambil detail: Agency Data Analyst di MSIG Life...
  -> Mengambil detail: Data Analyst (Regional BD) - Business Intelligence di Shopee...
  -> Mengambil detail: Business Intelligence di Monee...
  -> Mengambil detail: Data Analyst Associate di OttoDigital Group...
  -> Mengambil detail: Data Analyst di PT Bank Digital BCA (BCA Digital)...
  -> Mengambil detail: Business Data Analytics Lead di Amar Bank...
  -> Mengambil detail: Operations Analytics Associate di Grab...
  -> Mengambil detail: Data Management Analyst di PT. Permodalan Nasional Madani (Persero)...
  -> Mengambil detail: Data Analyst - MODA di PT Astra International Tbk...
  -> Mengambil detail: Data Analyst di Batumbu...
  -> Mengambil detail: Growth Data Analyst, ShopeePay di Monee...
  -> Mengambil detail: Data Analyst di Catalyst...
  -> Mengambil det

In [10]:
if len(data_lowongan) > 0:
    df = pd.DataFrame(data_lowongan)

    nama_file = 'raw_dataset_linkedin.csv'
    df.to_csv(nama_file, index=False)

    print(f"✅ Dataset berhasil disimpan sebagai {nama_file}")
    print(df.head(3))
else:
    print("⚠️ Belum ada data yang bisa disimpan. Cek kembali hasil dari Cell Debug.")

✅ Mantap! Dataset berhasil disimpan sebagai raw_dataset_linkedin.csv
                                 id                            job_title  \
0  74763aa4cfc4ffeefd0ea5e488e521d5  Data Analyst, Business Intelligence   
1  be23c0ed30e2b92628f9ef811e4e9109                         Data Analyst   
2  dbe7a93768caa2018b8cd141e30d456e                  Agency Data Analyst   

  company_name                 location       job_type experience_level  \
0       Shopee  Jakarta Raya, Indonesia  Tidak tertera    Tidak tertera   
1    Traveloka        Banten, Indonesia  Tidak tertera    Tidak tertera   
2    MSIG Life  Jakarta Raya, Indonesia  Tidak tertera    Tidak tertera   

               education_req                                    salary_range  \
0  Cek di kolom requirements  Tidak tertera (LinkedIn jarang publikasi gaji)   
1  Cek di kolom requirements  Tidak tertera (LinkedIn jarang publikasi gaji)   
2  Cek di kolom requirements  Tidak tertera (LinkedIn jarang publikasi gaji)   

    